# Module 297 — Jailbreaks II Worksheet

This notebook accompanies HTB Section 8. It explains the strategy families and implements the lesson's character-to-number encoding with harmless sample text.

It does not encode requests from the lab's unlawful examples, submit prompts, or make network requests. The encoding is a reversible representation, not encryption.

## Strategy map

Token smuggling changes how text is represented. A suffix adds text at the end of a prompt. An adversarial suffix is selected to influence a particular model and may not transfer to another. Opposite or sudo mode asks for an alternate persona. IMM gives the model an encoding description, asks it to use that encoding, and then supplies an encoded task.

These labels describe approaches to changing the model's response. They do not authenticate a user, grant permission, or guarantee success.

In [ ]:
# High-level descriptions only; no prompt payloads are stored or executed.
strategies = {
    "Token smuggling": "change the visible representation of the request",
    "Suffix": "append text that may shift likely continuations",
    "Adversarial suffix": "use a model-specific sequence selected to steer output",
    "Opposite/sudo mode": "ask for an alternate persona that ignores a prompt restriction",
    "IMM": "describe an encoding, supply an encoded task, and request encoded output",
}

for name, idea in strategies.items():
    print(f"{name}: {idea}")

## Read the encoding idea aloud

- **ord(c)** is read **“ord of c.”** It returns a number representing character c.
- **chr(n)** is read **“char of n.”** It returns the character represented by number n.
- **encode(text) = [ord(c) for each character c in text]** is read **“encode of text equals the list of ord of c for each character c in text.”** It turns each character into a number and collects the numbers in order.
- **decode(numbers) = join(chr(n) for each number n)** is read **“decode of numbers equals join of char of n for each number n.”** It turns the numbers back into characters and joins them into text.

For cat!, the mapping is c to 99, a to 97, t to 116, and ! to 33, giving [99, 97, 116, 33]. The null character with code point zero has numeric value zero, so the HTB example's subtract-zero/add-zero steps do not change the mapping.

The example is encoding, not encryption: anyone can reverse it. These values are Python Unicode code points; for basic ASCII characters they match ASCII codes. Unicode code points and UTF-8 bytes differ for many non-ASCII characters.

In [ ]:
def encode(text):
    """Return one integer code point for each character in text."""
    return [ord(character) for character in text]

def decode(numbers):
    """Turn character code points back into one text string."""
    return "".join(chr(number) for number in numbers)

# Use a harmless example so we can see each step.
plain_text = "cat!"
encoded_numbers = encode(plain_text)
decoded_text = decode(encoded_numbers)

print("Original text:", plain_text)
print("Encoded numbers:", encoded_numbers)
print("Decoded text:", decoded_text)

# This check demonstrates the round trip: decode(encode(text)) returns text.
assert decoded_text == plain_text

## The Haskell notation in the lesson

The type signature **encode :: String -> [Int]** is read **“encode has type string to list of integers.”** The double colon means “has type.” The arrow is read “to.” String is text; [Int] means a list of integer values.

The recursive pattern **c:cs** is read **“c cons cs.”** It means one first character c followed by the remaining characters cs. The function encodes that first character, then calls itself on the remainder. The empty string becomes an empty list. Python's list comprehension performs the same mapping in a shorter form.

## IMM as a multi-step transform

1. Explain the encoding to the model.
2. Ask it to respond using the same encoding.
3. Supply the task in encoded form.
4. The model must decode the task, create a response, and encode its answer.

This depends on the model correctly following several steps. HTB warns that the lab's target model may not be capable enough for IMM to work. A failure may mean the model could not follow the transformation, not that every encoding-based test is ineffective.

### Security connection

Encoding does not remove the meaning or impact of a request. A robust control should inspect or constrain what the model can do after any decoding stage and enforce permissions in application code. This notebook demonstrates only a harmless local conversion.